# ⚡ AI Flashcard Generator + Chatbot 🗨


In [9]:
!pip install -q "gradio==3.50.2" groq PyMuPDF
import importlib, gradio
importlib.reload(gradio)
print("Gradio version:", gradio.__version__)

Gradio version: 3.50.2


## Imports

In [10]:
import gradio as gr
from groq import Groq
import fitz
import json
import re
from google.colab import userdata

## Configure API Key

In [11]:
GROQ_API_KEY = userdata.get('GROQ_API_KEY')

client = Groq(api_key=GROQ_API_KEY)

MODELS = {
    "Llama 3.1 8B  (Fast)"   : "llama-3.1-8b-instant",
    "Llama 3.3 70B (Smart)"  : "llama-3.3-70b-versatile",
    "Llama 4 Scout (Latest)" : "meta-llama/llama-4-scout-17b-16e-instruct",
}

print("Groq ready!")

Groq ready!


## Chat Logic

In [12]:
def chat_with_groq(message, history, model_name):
    model_id = MODELS.get(model_name, "llama-3.1-8b-instant")
    messages = [{"role": "system", "content": "You are a helpful AI assistant. Be concise and clear."}]
    for human, assistant in history:
        messages.append({"role": "user",      "content": human})
        messages.append({"role": "assistant", "content": assistant})
    messages.append({"role": "user", "content": message})
    try:
        response = client.chat.completions.create(
            model=model_id,
            messages=messages,
            temperature=0.7,
            max_tokens=2048,
        )
        return response.choices[0].message.content
    except Exception as e:
        return "Error: " + str(e)

def respond(message, history, model_name):
    if not message.strip():
        return history, ""
    reply = chat_with_groq(message, history, model_name)
    history = history + [(message, reply)]
    return history, ""

## Flashcard Logic

In [13]:
def extract_pdf_text(pdf_file):
    if pdf_file is None:
        return ""
    doc = fitz.open(pdf_file.name)
    pages = [page.get_text() for page in doc]
    doc.close()
    return "\n\n".join(pages)


def build_prompt(content, num_cards, difficulty):
    prompt  = "You are an expert tutor. Generate exactly " + str(num_cards) + " flashcards from the text below.\n\n"
    prompt += "Difficulty: " + difficulty + "\n"
    prompt += "- Easy   -> definitions, basic recall\n"
    prompt += "- Medium -> application, cause-effect\n"
    prompt += "- Hard   -> synthesis, edge cases\n\n"
    prompt += "Return ONLY a valid JSON array. No markdown, no extra text.\n"
    prompt += "Keys per object: \"question\", \"answer\", \"hint\"\n\n"
    prompt += "TEXT:\n" + content[:8000]
    return prompt


def parse_cards(raw):
    raw = re.sub(r"```(?:json)?", "", raw).strip().rstrip("`").strip()
    start = raw.find("[")
    end   = raw.rfind("]")
    if start == -1 or end == -1:
        raise ValueError("No JSON array found. Raw: " + raw[:300])
    return json.loads(raw[start:end + 1])


def generate_flashcards(text_input, pdf_file, num_cards, difficulty, model_name):
    model_id = MODELS.get(model_name, "llama-3.1-8b-instant")
    content  = text_input.strip()
    if pdf_file is not None:
        content = (content + "\n\n" + extract_pdf_text(pdf_file)).strip()
    if not content:
        return [], "Please paste text, or upload a PDF."
    try:
        response = client.chat.completions.create(
            model=model_id,
            messages=[{"role": "user", "content": build_prompt(content, num_cards, difficulty)}],
            temperature=0.7,
            max_tokens=4096,
        )
        cards = parse_cards(response.choices[0].message.content)
    except json.JSONDecodeError as e:
        return [], "JSON parse error: " + str(e)
    except Exception as e:
        return [], "Error: " + str(e)
    if not cards:
        return [], "Model returned 0 cards. Try different input."
    return cards, str(len(cards)) + " flashcard(s) generated!"


def render_card(cards, index):
    if not cards:
        return "No cards yet.", "", "", ""
    card = cards[index]
    nav  = "Card " + str(index + 1) + " / " + str(len(cards))
    return nav, card["question"], card["answer"], card["hint"]

def prev_card(cards, index):
    if not cards:
        return index, "—", "—", "—", "—"
    index = (index - 1) % len(cards)
    nav, q, a, h = render_card(cards, index)
    return index, nav, q, a, h

def next_card(cards, index):
    if not cards:
        return index, "—", "—", "—", "—"
    index = (index + 1) % len(cards)
    nav, q, a, h = render_card(cards, index)
    return index, nav, q, a, h

def on_generate(text, pdf, num, diff, model_name):
    cards, msg = generate_flashcards(text, pdf, num, diff, model_name)
    if not cards:
        return cards, 0, msg, "—", "—", "—", "—"
    nav, q, a, h = render_card(cards, 0)
    return cards, 0, msg, nav, q, a, h

def export_cards(cards):
    if not cards:
        return None
    lines = []
    for i, c in enumerate(cards, 1):
        lines.append("--- Card " + str(i) + " ---")
        lines.append("Q: " + c["question"])
        lines.append("A: " + c["answer"])
        lines.append("Hint: " + c["hint"])
        lines.append("")
    path = "/tmp/flashcards.txt"
    with open(path, "w") as f:
        f.write("\n".join(lines))
    return path